<a href="https://colab.research.google.com/github/chinh-hoangduc/QuantEcon-exercises/blob/main/sandbox_misallocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
!pip install jax

In [55]:
from typing import NamedTuple
import numpy as np
from scipy.stats import multivariate_normal, norm
import matplotlib.pyplot as plt
import jax.numpy as jnp
import jax.scipy as jsp
import jax
import quantecon as qe
from functools import partial

First, we define the class for households.

This one includes age, asset grid, initial endowment (`θ_grid`, `P_grid`), and the weights/probability of being in each pair of endowment.

In [94]:
class Household(NamedTuple):
    T: int
    a_grid: jnp.ndarray
    θ_grid: jnp.ndarray
    P_grid: jnp.ndarray
    endow_wts: jnp.ndarray

def create_household(
    T: int = 22,
    a_min: float = 0, a_num: int = 150,
    θ_num: int = 9, P_num: int = 25,
    σ_θ: float = 0.4, σ_P: float = 0.5, ρ: float = 0.4,
    n_std: int = 4,
) -> Household:

    a_max = jnp.exp(n_std * σ_P) + 1   # align wealth grid of generations
    a_grid = jnp.linspace(a_min, a_max, a_num)


    def create_endowment_grid(θ_num, P_num, σ_θ, σ_P, ρ, n_std):
        # 1. Marginal nodes: equally spaced in log space (over ±n_std * sigma)
        log_θ = jnp.linspace(-n_std * σ_θ, n_std * σ_θ, θ_num)
        log_P = jnp.linspace(-n_std * σ_P, n_std * σ_P, P_num)

        # 2. Cell boundaries: midpoints, extended to ±inf at the tails
        def edges(x):
            mids = jnp.asarray(0.5 * (x[:-1] + x[1:]))
            return jnp.concatenate([jnp.array([-jnp.inf]), mids, jnp.array([jnp.inf])])
        e_θ = edges(log_θ)   # length n_theta + 1
        e_P = edges(log_P)   # length n_P + 1

        # 3. Cell probabilities from the bivariate normal CDF
        Sigma = np.array([[σ_θ**2, ρ * σ_θ * σ_P],
                          [ρ * σ_θ * σ_P, σ_P**2]])
        rv = multivariate_normal(mean=[0.0, 0.0], cov=Sigma)

        # CDF at each corner, then inclusion–exclusion for the rectangle mass
        T, P = np.meshgrid(e_θ, e_P, indexing='ij')          # (n_theta+1, n_P+1)
        C = rv.cdf(np.stack([T.ravel(), P.ravel()], axis=-1)).reshape(T.shape)
        W = C[1:,1:] - C[:-1,1:] - C[1:,:-1] + C[:-1,:-1]        # (n_theta, n_P)
        W = W / W.sum()

        # 4. Return levels (not logs) plus the joint weight matrix
        θ_grid = jnp.asarray(np.exp(log_θ))
        P_grid = jnp.asarray(np.exp(log_P))
        endow_wts = jnp.asarray(W)
        return θ_grid, P_grid, endow_wts

    θ_grid, P_grid, endow_wts = create_endowment_grid(θ_num, P_num, σ_θ, σ_P, ρ, n_std)


    return Household(T, a_grid, θ_grid, P_grid, endow_wts)

In [97]:
household = create_household()

Then, we define the class for occupation-specific shocks on human capital accumulation.

People in different occupations will receive different shock processes.

In this simple model version, that coincides with two educational level since there are only two occupations.

In [90]:
class Shock(NamedTuple):
    ε_grid: jnp.ndarray
    Π: jnp.ndarray
    μ_ε: jnp.ndarray

def create_shock(ρ: float, σ_η: float, n: int = 7) -> Shock:
    mc = qe.markov.approximation.tauchen(n, ρ, σ_η)
    Π = jnp.asarray(mc.P)
    ε_grid = jnp.asarray(mc.state_values)

    # Stationary distribution: left eigenvector of Pi with eigenvalue 1
    μ_ε = jnp.asarray(mc.stationary_distributions[0])
    return Shock(ε_grid, Π, μ_ε)

In [91]:
shock_1 = create_shock(ρ=0.8836, σ_η=0.2744, n=7)   # Both education level can do job 1
shock_2 = create_shock(ρ=0.9216, σ_η=0.2356, n=7)   # Only college graduate can do job 2

We then define firms.

In [ ]:
class Firm(NamedTuple):
    α: float
    A1: float
    A2: float
    σ: float
    δ: float

def create_firm(
    α: float = 0.36,
    A1: float = 1,
    A2: float = 1.5,
    σ: float = 2,
    δ: float = 0.11,
) -> Firm:
    return Firm(α, A1, A2, σ, δ)

In [ ]:
firm = create_firm()

Then we define some functions for preferences and the factor prices.

In [ ]:
γ = 1.5

def bequest(a, abar = 5.0, ψ = 2.0):
    return ψ * (a + abar)**(1 - γ) / (1 - γ)

def u(c):
    return c**(1 - γ) /

In [ ]:
def interest(L1, L2, K, firm: Firm):
    α, A1, A2, σ, δ = firm
    Lbar = (A1 * L1 ** ((σ - 1)/σ) + A2 * L2 ** ((σ - 1)/σ)) ** (σ/(σ - 1))
    return α * (Lbar / K) ** (1 - α) - δ

def wages(L1, L2, K, firm: Firm):
    α, A1, A2, σ, δ = firm
    Lbar = (A1 * L1 ** ((σ - 1)/σ) + A2 * L2 ** ((σ - 1)/σ)) ** (σ/(σ - 1))
    Y = K**α * Lbar**(1-α)
    w1 = (1 - α) * Y/Lbar * A1 * (Lbar/L1)**(1/σ)
    w2 = (1 - α) * Y/Lbar * A2 * (Lbar/L2)**(1/σ)
    return w1, w2

Functional form for initial transfers.

In [ ]:
def transfer(household, τ0 = 1., τ1 = 0.5, ϕ0 = 0.7, ϕ1 = 1.):
    T, a_grid, θ_grid, P_grid, endow_wts = household
    τN = τ0 * P_grid ** ϕ0
    τC = τN + τ1 * P_grid ** ϕ1
    return τN, τC

Functional form for skill upgrade.

In [ ]:
def upskill(household, ζ1 = 0.15, ζ2 = 0.05, ζ3 = 0.03):
    T, a_grid, θ_grid, P_grid, endow_wts = household
    log_θ = jnp.log(θ_grid)
    log_P = jnp.log(P_grid)
    return jnp.exp(log_θ + ζ1 + ζ2 * log_P + ζ3 * log_θ * log_P)

Exogenous probability of dropout.

In [ ]:
def dropout(household, δ1 = 0.6, δ2 = 0.8, δ3 = 0.5, δ4 = 0.2):
    T, a_grid, θ_grid, P_grid, endow_wts = household
    log_θ = jnp.log(θ_grid)
    log_P = jnp.log(P_grid)
    return 1 / (1 + jnp.exp(δ1 + δ2 * log_θ + δ3 * log_P + δ4 * log_θ * log_P))

Now we solve for the working life problem by backward induction.

The non-college type is the easiest, we will start from there.

We will solve it by DSDP, which requires defining `Q` and `R` first.

In [ ]:
def Q():
    return Q

def R():
    return R